# 📈 Notebook 2: The Phi Accrual Failure Detector

Hayashibara's idea (used by **Cassandra**, **Akka**): keep a sliding window of recent heartbeat *intervals*, model them as a normal distribution, and compute

$$\phi(t) = -\log_{10}\big( P(\text{interval} \geq t - t_{last}) \big)$$

`phi` rises smoothly the longer we go without a heartbeat. Each application picks its own threshold (Cassandra defaults to 8).


## 🛠️ Setup

```bash
cd 02-distributed-primitives/phi-accrual-failure-detection
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 Implementation

In [ ]:
import math, random
from collections import deque

class PhiAccrual:
    def __init__(self, window=200, min_std=0.1):
        self.window = deque(maxlen=window)
        self.last = None
        self.min_std = min_std

    def heartbeat(self, t):
        if self.last is not None:
            self.window.append(t - self.last)
        self.last = t

    def phi(self, t):
        if not self.window or self.last is None:
            return 0.0
        mean = sum(self.window) / len(self.window)
        var = sum((x-mean)**2 for x in self.window) / len(self.window)
        std = max(math.sqrt(var), self.min_std)
        # Tail probability of a normal distribution, using complementary error function
        gap = t - self.last
        z = (gap - mean) / std
        p = 0.5 * math.erfc(z / math.sqrt(2))
        if p <= 0:
            return 50.0
        return -math.log10(p)


## 📊 Replay a heartbeat stream and watch phi rise

In [ ]:
import matplotlib.pyplot as plt
random.seed(7)
INTERVAL, JITTER, TOTAL, DEAD_AT = 1.0, 0.4, 30.0, 20.0
beats, now = [], 0.0
while now < TOTAL:
    now += INTERVAL + random.uniform(-JITTER, JITTER)
    if now >= DEAD_AT: break
    beats.append(now)

det = PhiAccrual()
ts, phis = [], []
i = 0
t = 0.0
while t < TOTAL:
    while i < len(beats) and beats[i] <= t:
        det.heartbeat(beats[i]); i += 1
    ts.append(t); phis.append(det.phi(t))
    t += 0.1

plt.figure(figsize=(9,3))
plt.plot(ts, phis)
plt.axhline(8, color='red', linestyle='--', label='threshold=8 (Cassandra default)')
plt.axvline(DEAD_AT, color='black', linestyle=':', label='node truly died')
plt.xlabel('time (s)'); plt.ylabel('phi'); plt.legend(); plt.title('Phi over time')
plt.show()


## 🧠 Why phi wins

| | Fixed timeout | Phi accrual |
|---|---|---|
| Adapts to network | no | yes (learns mean/variance) |
| One value fits all apps | no — every app shares it | each app picks its own threshold |
| False positives | many under jitter | rare; suspicion grows smoothly |
| Detection speed | tied to timeout | tied to threshold *and* network calmness |

Pick a low threshold (~5) for fast-but-jumpy detection (e.g. UI status). Pick a high one (~12) for safety-critical decisions like leader fencing.